In [ ]:
# TODO: if working, create mamba version 
# TODO: clean up code
# TODO: any bugs?
# TODO: is dataset split wrapper a good pattern?
# TODO: review reshapes vs views
# TODO: review permutes
# TODO: go deeper by removing F. and nn layers

In [ ]:
CONFIG = {
    "block_size" : 64,
    "n_embed" : 128,
    "batch_size" : 64,
    "hidden_size" : 256,
    "n_epochs" : 50,
    "learning_rate" : 3e-3,
}

In [ ]:
with open("data/lotr.txt", "r", encoding="utf-8") as f: TEXT = f.read()
TEXT[:10000]

In [ ]:
VOCAB = sorted(list(set(TEXT)))
CONFIG["vocab_size"] = len(VOCAB)
ctoi = dict((c, i) for i, c in enumerate(VOCAB))
itoc = dict((i, c) for i, c in enumerate(VOCAB))
encode = lambda str: [ctoi[c] for c in str]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
decode(encode("hello world"))

In [ ]:
TOKENS = encode(TEXT)
TOKENS[:10]

In [ ]:
import torch
from torch.utils.data import Dataset

class ChunkedDataset(Dataset):
    def __init__(self, tokens, block_size=CONFIG["block_size"]):
        tokens = torch.tensor(tokens)

        n_chunks = (len(tokens)) // (block_size + 1)
        tokens = tokens[:n_chunks * (block_size + 1)]

        self.chunks = tokens.view(n_chunks, block_size + 1)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

    def __len__(self):
        return len(self.chunks)

full_dataset = ChunkedDataset(TOKENS)
full_dataset[0]

In [ ]:
class DatasetSplit(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices
    
    def __getitem__(self, idx):
        dataset_idx = self.indices[idx]
        item = self.dataset[dataset_idx]
        return item
        
    def __len__(self):
        return len(self.indices)

In [ ]:
import random
indices = list(range(len(full_dataset)))
random.shuffle(indices)
split_idx = int(len(indices) * 0.8)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]
train_dataset = DatasetSplit(full_dataset, train_indices)
val_dataset = DatasetSplit(full_dataset, val_indices)

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=CONFIG["batch_size"]
)
for x, y in train_dataloader:
    print("X:", x.shape)
    print("Y:", y.shape)
    break

In [ ]:
from torch.utils.data import DataLoader

val_dataloader = DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=CONFIG["batch_size"]
)
for x, y in val_dataloader:
    print("X:", x.shape)
    print("Y:", y.shape)
    break

In [ ]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(
        self, 
        vocab_size=CONFIG["vocab_size"], 
        n_embed=CONFIG["n_embed"], 
        hidden_size=CONFIG["hidden_size"]
    ):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.embeddings = nn.Embedding(vocab_size, n_embed) # TODO: do manually
        self.W_x = nn.Linear(n_embed, hidden_size)#, bias=False) # TODO: do manually
        self.W_h = nn.Linear(hidden_size, hidden_size)#, bias=False)

        #self.bias = nn.Parameter(torch.zeros(hidden_size))

        self.head = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        B, T = x.shape
        x_emb = self.embeddings(x) # (B, T) -> (B, T, C)
        x_transf = self.W_x(x_emb) # (B, T, C) -> (B, T, H)
        ht_prev = torch.zeros(B, self.hidden_size) # (B, H)
        outs = []
        for t in range(T):
            xt = x_transf[:,t,:] # (B, T, H) -> (B, H)
            ht = self.W_h(ht_prev) # (B, H) -> (B, H)
            out = torch.tanh(xt + ht) # (B, H) + (B, H) -> (B, H)
            outs.append(out)
            ht_prev = out
        outs = torch.stack(outs) # (B, T, H)
        logits = self.head(outs) # (B, T, H) -> (B, T, C)
        logits = logits.permute(1, 0, 2)
        return logits
    
    # TODO: expected this to be faster but its not, investigate
    def _forward(self, x):
        B, T = x.shape
        x_emb = self.embeddings(x)
        x_transf = self.W_x(x_emb)
        h = torch.zeros(B, T, self.hidden_size, device=x.device)  # Add device
        h_prev = torch.zeros(B, self.hidden_size, device=x.device)
        for t in range(T):
            h_t = torch.tanh(x_transf[:, t] + self.W_h(h_prev))  # Compute first
            h[:, t] = h_t  # Then assign (separate operations)
            h_prev = h_t   # Then update
        logits = self.head(h)
        return logits

model = RNN()
x, _ = next(iter(train_dataloader)) # TODO: what does this mean?
logits = model(x)
x.shape, logits.shape

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses = []
    for x, y in tqdm(loader, disable=True):
        logits = model(x)
        logits = logits.reshape(-1, logits.size(-1)) # (B,T,V) -> (B*T,V)
        y = y.reshape(-1) # (B,T) -> (B*T)
        loss = F.cross_entropy(logits, y) # TODO: do manually
        loss_i = loss.item()
        losses.append(loss_i)
    loss = sum(losses) / len(loader)
    return loss

evaluate(model, val_dataloader)

In [ ]:
import torch.optim
import torch.nn.functional as F

n_epochs = CONFIG["n_epochs"]
lr = CONFIG["learning_rate"]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
train_losses = []
val_losses = []
train_loss = None
val_loss = None
postfix = dict(train_loss=train_loss,val_loss=val_loss)
patience = max_patience = 3
for _ in range(n_epochs):
    model.train()
    losses = []
    pbar = tqdm(train_dataloader)
    for x, y in pbar:
        logits = model(x)
        # TODO: why view doesnt work?
        logits = logits.reshape(-1, logits.size(-1)) # (B,T,V) -> (B*T,V)
        y = y.reshape(-1) # (B,T) -> (B*T)
        loss = F.cross_entropy(logits, y) # TODO: do manually
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_i = loss.item()
        losses.append(loss_i)
        _postfix = dict([(k,v) for k,v in postfix.items() if v is not None])
        pbar.set_postfix(_postfix)
    val_loss = evaluate(model, val_dataloader)
    train_loss = sum(losses) / len(losses)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    postfix["train_loss"] = train_loss
    postfix["val_loss"] = val_loss
    postfix["patience"] = patience
    _postfix = dict([(k,v) for k,v in postfix.items() if v is not None])
    pbar.set_postfix(_postfix)

    if len(val_losses) > 1:
        val_loss_delta = val_losses[-2] - val_losses[-1]
        if val_loss_delta > 0.01: patience = max_patience
        else: 
            patience -= 1
            print(f"Patience = {patience}, val_loss_delta = {val_loss_delta}")
        if patience == 0: 
            print("Early stopping due to val_loss stall")
            n_epochs = 0
            break


In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_losses) # TODO: manually plot gradnorms too
plt.plot(val_losses)

In [ ]:
def generate(model, prompt, max_len=100):
    tokens = encode(prompt)
    max_len = max_len - len(tokens)
    assert max_len > 0
    print(prompt, end="")
    for _ in range(max_len):
        tokens_t = torch.tensor(tokens).unsqueeze(0)
        logits = model(tokens_t) # (B, T) -> (B, T, C)
        logits = logits[:,-1, :] # (B, T, C) -> (B, C)
        probs = F.softmax(logits, dim=-1) # (B, C) -> (B, P)
        token = torch.multinomial(probs, num_samples=1).squeeze().item() # (B, P) -> (B,)
        tokens.append(token)
        token_s = decode([token])
        print(token_s, end="")

generate(model, "Frodo picked up")